# gemini_2_5_flash + panns Multimodal Evaluation (eval)

Queries condition C SQL on combined DB. Writes xlsx to data/analysis_gemini_2_5_flash_panns/.


In [1]:
import sys, os, sqlite3, json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
for p in [ROOT] + list(ROOT.parents):
    if (p / '.gitignore').exists():
        ROOT = p; break
sys.path.insert(0, str(ROOT / 'backend/src'))
os.environ['PROJECT_ROOT'] = str(ROOT)
ABLATION_DIR = ROOT / 'data' / 'ablation_gemini_2_5_flash_panns'
ANALYSIS_DIR = ROOT / 'data' / 'analysis_gemini_2_5_flash_panns'
VIDEO_DIR = ROOT / 'data/videos/eval'
GT_PATH = ROOT / 'data/videos/eval/ground_truth.xlsx'
from service.impl.events_service_impl import queries_for_condition
from utils.database import setup_database
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {ROOT}')
print(f'Analysis dir: {ANALYSIS_DIR}')


Project root: /home/ghiffaryr/iseql/multimodal-surveillance-iseql
Analysis dir: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_gemini_2_5_flash_panns


In [2]:
# ── Audio configs ──
AUDIO_CONFIGS = [
    ('w1_0s_h1_0s', 1.0, 1.00, 'ab0'),
    ('w1_0s_h0_5s', 1.0, 0.50, 'ab1'),
    ('w2_5s_h2_5s', 2.5, 2.50, 'ab2'),
    ('w2_5s_h1_25s', 2.5, 1.25, 'ab3'),
    ('w5_0s_h5_0s', 5.0, 5.00, 'ab4'),
    ('w5_0s_h2_5s', 5.0, 2.50, 'ab5'),
    ('w10_0s_h10_0s', 10.0, 10.00, 'ab6'),
    ('w10_0s_h5_0s', 10.0, 5.00, 'ab7'),
]

# ── Source DBs ──
visual_db = ROOT / 'data' / 'ablation_gemini_2_5_flash' / 'gemini_2_5_flash.db'
if not visual_db.exists():
    raise FileNotFoundError(f'Visual DB missing: {visual_db}')
missing = []
for win_label, window, hop, prefix in AUDIO_CONFIGS:
    audio_db = ROOT / 'data' / 'ablation_panns' / f'panns_{win_label}.db'
    if not audio_db.exists():
        missing.append(str(audio_db))
if missing:
    raise FileNotFoundError(f'Audio DBs missing: {missing}')
print(f'Visual DB: {visual_db}')
print(f'{len(AUDIO_CONFIGS)} audio configs: {[c[0] for c in AUDIO_CONFIGS]}')


Visual DB: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_gemini_2_5_flash/gemini_2_5_flash.db
8 audio configs: ['w1_0s_h1_0s', 'w1_0s_h0_5s', 'w2_5s_h2_5s', 'w2_5s_h1_25s', 'w5_0s_h5_0s', 'w5_0s_h2_5s', 'w10_0s_h10_0s', 'w10_0s_h5_0s']


In [3]:
# ── Create combined DB for each audio config ──
ablation_dir = ROOT / 'data' / 'ablation_gemini_2_5_flash_panns'
ablation_dir.mkdir(parents=True, exist_ok=True)
for win_label, window, hop, prefix in AUDIO_CONFIGS:
    audio_db = ROOT / 'data' / 'ablation_panns' / f'panns_{win_label}.db'
    combined_db = ablation_dir / f'gemini_2_5_flash_panns_{win_label}.db'
    if combined_db.exists():
        print(f'Combined DB exists: {combined_db}')
        continue
    conn, cur = setup_database(combined_db)
    conn.execute('PRAGMA journal_mode=WAL')
    audio_conn = sqlite3.connect(str(audio_db))
    adf = pd.read_sql_query('SELECT * FROM SoundPerInterval', audio_conn)
    audio_conn.close()
    adf['AnalysisID'] = adf['AnalysisID'].apply(
        lambda x: f'gemini_2_5_flash_panns_s{x.split("_")[-1][1:].split(".")[0]}' if 'ab' in x else x)
    adf.to_sql('SoundPerInterval', conn, if_exists='append', index=False)
    visual_conn = sqlite3.connect(str(visual_db))
    vdf = pd.read_sql_query('SELECT * FROM VisualPerInterval', visual_conn)
    participant_df = pd.read_sql_query('SELECT * FROM VisualParticipant', visual_conn)
    perframe_df = pd.read_sql_query('SELECT * FROM VisualPerFrame', visual_conn)
    rel_df = pd.read_sql_query('SELECT * FROM VisualRelation', visual_conn)
    visual_conn.close()
    vdf['AnalysisID'] = vdf['AnalysisID'].apply(
        lambda x: f'gemini_2_5_flash_panns_s{x.split("_")[-1][1:].split(".")[0]}' if x.startswith('gemini_2_5_flash_') else x)
    perframe_df['AnalysisID'] = perframe_df['AnalysisID'].apply(
        lambda x: f'gemini_2_5_flash_panns_s{x.split("_")[-1][1:].split(".")[0]}' if x.startswith('gemini_2_5_flash_') else x)
    rel_df['AnalysisID'] = rel_df['AnalysisID'].apply(
        lambda x: f'gemini_2_5_flash_panns_s{x.split("_")[-1][1:].split(".")[0]}' if x.startswith('gemini_2_5_flash_') else x)
    conn.execute('DELETE FROM VisualPerInterval')
    vdf.to_sql('VisualPerInterval', conn, if_exists='append', index=False)
    participant_df.to_sql('VisualParticipant', conn, if_exists='append', index=False)
    perframe_df.to_sql('VisualPerFrame', conn, if_exists='append', index=False)
    rel_df.to_sql('VisualRelation', conn, if_exists='append', index=False)
    conn.commit()
    size_kb = os.path.getsize(combined_db) // 1024
    print(f'Combined DB created: {combined_db} ({size_kb} KiB)')
    conn.close()
print('All combined DBs ready.')


Combined DB created: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_gemini_2_5_flash_panns/gemini_2_5_flash_panns_w1_0s_h1_0s.db (4 KiB)
Combined DB created: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_gemini_2_5_flash_panns/gemini_2_5_flash_panns_w1_0s_h0_5s.db (4 KiB)
Combined DB created: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_gemini_2_5_flash_panns/gemini_2_5_flash_panns_w2_5s_h2_5s.db (4 KiB)


Combined DB created: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_gemini_2_5_flash_panns/gemini_2_5_flash_panns_w2_5s_h1_25s.db (4 KiB)
Combined DB created: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_gemini_2_5_flash_panns/gemini_2_5_flash_panns_w5_0s_h5_0s.db (4 KiB)


Combined DB created: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_gemini_2_5_flash_panns/gemini_2_5_flash_panns_w5_0s_h2_5s.db (4 KiB)
Combined DB created: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_gemini_2_5_flash_panns/gemini_2_5_flash_panns_w10_0s_h10_0s.db (4 KiB)


Combined DB created: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/ablation_gemini_2_5_flash_panns/gemini_2_5_flash_panns_w10_0s_h5_0s.db (4 KiB)
All combined DBs ready.


In [4]:
# ── Load GT ──
gt = pd.read_excel(GT_PATH, sheet_name='Ground Truth')
gt_visual = gt[gt['modality'] == 'visual'].dropna(subset=['scene'])
gt_visual['scene'] = gt_visual['scene'].astype(int)
gt_audio = gt[gt['modality'] == 'audio'].dropna(subset=['scene'])
gt_audio['scene'] = gt_audio['scene'].astype(int)
gt_audio['class'] = gt_audio['class'].apply(lambda c: c.strip().lower().replace(' ', '_') if isinstance(c, str) else c)
expected_df = pd.read_excel(GT_PATH, sheet_name='Expected Events')
print(f'GT: visual={len(gt_visual)} audio={len(gt_audio)} expected={len(expected_df)}')


GT: visual=47 audio=40 expected=30


In [5]:
# ── DELTAS for condition C ──
DELTAS = {
    "delta_visual_vehicle_escape": 50,
    "delta_visual_loitering": 150,
    "delta_visual_handoff": 240,
    "delta_sound_fight": 120,
    "delta_sound_vehicle_escape": 150,
    "delta_sound_vehicle_collision": 60,
}


In [6]:
# ── Multimodal event evaluation (all audio configs) ──
audio_evts = set(expected_df[expected_df['audio'].notna() & (expected_df['audio'] != '')]['event'].unique())
all_scenes = sorted(expected_df['scene'].unique())
summary_rows = []

for win_label, window, hop, prefix in AUDIO_CONFIGS:
    combined_db = ablation_dir / f'gemini_2_5_flash_panns_{win_label}.db'
    conn = sqlite3.connect(str(combined_db))
    event_rows = []
    for _, row in expected_df.iterrows():
        aid = f'gemini_2_5_flash_panns_s{row["scene"]}'
        evt = row['event']
        try:
            sql_map = queries_for_condition('C', DELTAS, analysis_id=aid)
            sql = sql_map.get(evt, 'SELECT 0 WHERE 1=0')
            df = pd.read_sql_query(sql, conn)
            det = not df.empty
            result = 'TP' if det else 'FN'
        except Exception as e:
            print(f'  {evt} query failed: {e}')
            det = False; result = 'ERROR'
        parts = []
        if evt in audio_evts:
            all_audio = conn.execute('SELECT SoundClass, StartFrame, EndFrame FROM SoundPerInterval WHERE AnalysisID = ?', (aid,)).fetchall()
            if all_audio:
                parts += [f'sound: {s}({sf}-{ef})' for s, sf, ef in all_audio]
        all_visual = conn.execute('SELECT RelationType, StartFrame, EndFrame FROM VisualPerInterval WHERE AnalysisID = ?', (aid,)).fetchall()
        if all_visual:
            parts += [f'visual: {r}({sf}-{ef})' for r, sf, ef in all_visual]
        if det and not parts:
            parts = [f'{evt} (query matched)']
        event_rows.append({'scene': row['scene'], 'event': evt, 'detected': 'YES' if det else 'NO', 'result': result, 'relations': ', '.join(parts)})

    # FP pass
    for evt in sorted(expected_df['event'].unique()):
        pos_scenes = set(expected_df[expected_df['event'] == evt]['scene'])
        for scene in all_scenes:
            if scene in pos_scenes:
                continue
            aid = f'gemini_2_5_flash_panns_s{scene}'
            try:
                sql_map = queries_for_condition('C', DELTAS, analysis_id=aid)
                sql = sql_map.get(evt, 'SELECT 0 WHERE 1=0')
                df = pd.read_sql_query(sql, conn)
                if not df.empty:
                    parts = []
                    if evt in audio_evts:
                        all_audio = conn.execute('SELECT SoundClass, StartFrame, EndFrame FROM SoundPerInterval WHERE AnalysisID = ?', (aid,)).fetchall()
                        if all_audio:
                            parts += [f'sound: {s}({sf}-{ef})' for s, sf, ef in all_audio]
                    all_visual = conn.execute('SELECT RelationType, StartFrame, EndFrame FROM VisualPerInterval WHERE AnalysisID = ?', (aid,)).fetchall()
                    if all_visual:
                        parts += [f'visual: {r}({sf}-{ef})' for r, sf, ef in all_visual]
                    event_rows.append({'scene': scene, 'event': evt, 'detected': 'YES', 'result': 'FP', 'relations': ', '.join(parts) if parts else '|'})
            except:
                pass
    vpi = conn.execute('SELECT COUNT(*) FROM VisualPerInterval').fetchone()[0]
    spi = conn.execute('SELECT COUNT(*) FROM SoundPerInterval').fetchone()[0]
    conn.close()

    edf = pd.DataFrame(event_rows)
    edf['relations'] = edf['relations'].fillna('')
    metrics = []
    for evt in sorted(expected_df['event'].unique()):
        sub = edf[edf['event'] == evt]
        tpp = len(sub[sub['result'] == 'TP']); fpp = len(sub[sub['result'] == 'FP']); fnn = len(sub[sub['result'] == 'FN'])
        support = tpp + fnn
        p = tpp / (tpp + fpp) if (tpp + fpp) > 0 else 0.0
        r = tpp / (tpp + fnn) if (tpp + fnn) > 0 else 0.0
        f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0
        metrics.append({'event': evt, 'precision': round(p,3), 'recall': round(r,3), 'f1': round(f1,3), 'TP': tpp, 'FP': fpp, 'FN': fnn, 'support': support})
    metrics_df = pd.DataFrame(metrics)
    xlsx_path = ANALYSIS_DIR / f'multimodal_event_eval_gemini_2_5_flash_panns_{win_label}.xlsx'
    with pd.ExcelWriter(xlsx_path) as writer:
        metrics_df.to_excel(writer, sheet_name='Summary', index=False)
        for sc in all_scenes:
            sc_df = edf[edf['scene'] == sc]
            if not sc_df.empty:
                sc_df.to_excel(writer, sheet_name=f'Scene_{sc}', index=False)
    print(f'Written: {xlsx_path}')

    tp = len(edf[edf['result'] == 'TP']); fp = len(edf[edf['result'] == 'FP']); fn = len(edf[edf['result'] == 'FN'])
    support = tp + fn
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    summary_rows.append({'visual': 'gemini_2_5_flash', 'reid': True, 'audio': 'panns', 'window': window, 'hop': hop,
        'precision': round(precision,3), 'recall': round(recall,3), 'f1': round(f1,3),
        'TP': tp, 'FP': fp, 'FN': fn, 'support': support, 'VPI': vpi, 'SPI': spi})
    print(f'  [{win_label}] Summary: P={precision:.3f} R={recall:.3f} F1={f1:.3f} TP={tp} FP={fp} FN={fn}')

summary_df = pd.DataFrame(summary_rows)
summary_df.to_excel(ANALYSIS_DIR / 'summary.xlsx', index=False)
print(f'\n=== Pair summary (all configs) ===')
print(summary_df.to_string(index=False))


Written: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_gemini_2_5_flash_panns/multimodal_event_eval_gemini_2_5_flash_panns_w1_0s_h1_0s.xlsx
  [w1_0s_h1_0s] Summary: P=0.818 R=0.600 F1=0.692 TP=18 FP=4 FN=12


Written: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_gemini_2_5_flash_panns/multimodal_event_eval_gemini_2_5_flash_panns_w1_0s_h0_5s.xlsx
  [w1_0s_h0_5s] Summary: P=0.818 R=0.600 F1=0.692 TP=18 FP=4 FN=12


Written: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_gemini_2_5_flash_panns/multimodal_event_eval_gemini_2_5_flash_panns_w2_5s_h2_5s.xlsx
  [w2_5s_h2_5s] Summary: P=0.783 R=0.600 F1=0.679 TP=18 FP=5 FN=12


Written: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_gemini_2_5_flash_panns/multimodal_event_eval_gemini_2_5_flash_panns_w2_5s_h1_25s.xlsx
  [w2_5s_h1_25s] Summary: P=0.792 R=0.633 F1=0.704 TP=19 FP=5 FN=11


Written: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_gemini_2_5_flash_panns/multimodal_event_eval_gemini_2_5_flash_panns_w5_0s_h5_0s.xlsx
  [w5_0s_h5_0s] Summary: P=0.857 R=0.600 F1=0.706 TP=18 FP=3 FN=12


Written: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_gemini_2_5_flash_panns/multimodal_event_eval_gemini_2_5_flash_panns_w5_0s_h2_5s.xlsx
  [w5_0s_h2_5s] Summary: P=0.783 R=0.600 F1=0.679 TP=18 FP=5 FN=12


Written: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_gemini_2_5_flash_panns/multimodal_event_eval_gemini_2_5_flash_panns_w10_0s_h10_0s.xlsx
  [w10_0s_h10_0s] Summary: P=0.850 R=0.567 F1=0.680 TP=17 FP=3 FN=13


Written: /home/ghiffaryr/iseql/multimodal-surveillance-iseql/data/analysis_gemini_2_5_flash_panns/multimodal_event_eval_gemini_2_5_flash_panns_w10_0s_h5_0s.xlsx
  [w10_0s_h5_0s] Summary: P=0.850 R=0.567 F1=0.680 TP=17 FP=3 FN=13

=== Pair summary (all configs) ===
          visual  reid audio  window   hop  precision  recall    f1  TP  FP  FN  support  VPI  SPI
gemini_2_5_flash  True panns     1.0  1.00      0.818   0.600 0.692  18   4  12       30  114   32
gemini_2_5_flash  True panns     1.0  0.50      0.818   0.600 0.692  18   4  12       30  114   31
gemini_2_5_flash  True panns     2.5  2.50      0.783   0.600 0.679  18   5  12       30  114   28
gemini_2_5_flash  True panns     2.5  1.25      0.792   0.633 0.704  19   5  11       30  114   31
gemini_2_5_flash  True panns     5.0  5.00      0.857   0.600 0.706  18   3  12       30  114   19
gemini_2_5_flash  True panns     5.0  2.50      0.783   0.600 0.679  18   5  12       30  114   21
gemini_2_5_flash  True panns    10.0 10.00